# 01 · KiTS 2023 데이터 준비
Linux NVIDIA 서버 전용입니다. 로컬 개발 컴퓨터에서는 실행하지 않습니다.
먼저 README의 `setup_server.sh`를 실행하고 그 venv에서 Jupyter를 시작하세요.
준비 과정: 공식 학습용 489건 다운로드 → 체크섬 검증 → nnU-Net 변환 → 무결성 검사 → 3D full-resolution 전처리 → 5-fold 고정.
데이터는 CC BY-NC-SA 조건이며 [공식 안내](https://github.com/neheller/kits23#license-and-attribution)를 따릅니다.
코드/데이터 버전을 기록하지만 논문 당시 환경과의 완전한 동일성은 아직 확인되지 않았습니다.

In [ ]:
from pathlib import Path
import sys, os, json, shlex, subprocess
sys.dont_write_bytecode = True
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "nnunetv2").is_dir())
EXPERIMENT = REPO / "experiments/kits2023"
sys.path.insert(0, str(EXPERIMENT / "scripts"))
from workflow import read_config, paths, guard, environment, model_dir
CONFIG = EXPERIMENT / "config.server.json"
if not CONFIG.exists():
    CONFIG = EXPERIMENT / "config.json"  # 로컬 기본 설정: 실행 비활성화
cfg = read_config(CONFIG)
print("Config:", CONFIG)
print("Execution enabled:", cfg["execution_enabled"])
print("Storage:", cfg["storage_root"])


## 서버 확인
GPU, RAM, 저장 공간을 확인합니다. 전처리 worker 수는 config.server.json에서 조절합니다.

In [ ]:
guard(cfg)
import shutil
print(subprocess.check_output(["nvidia-smi"], text=True))
print("Disk:", shutil.disk_usage(Path(cfg["storage_root"])))
print(subprocess.check_output(["free", "-h"], text=True))


## 다운로드부터 전처리까지 시작
아래 셀은 `tmux`에서 준비 작업을 시작합니다. `tmux`가 없으면 서버에서 설치하세요.
브라우저/커널 종료와 무관하게 작업이 유지됩니다. 동일 세션이 있으면 다시 시작하지 않습니다.
중단 후 같은 셀을 다시 실행하면 검증된 다운로드/완료된 전처리는 재사용합니다.

In [ ]:
guard(cfg)
from workflow import paths
p = paths(cfg)
session = "kits23-prepare"
log = p["root"] / "prepare.console.log"
if subprocess.run(["tmux", "has-session", "-t", session], capture_output=True).returncode == 0:
    print("Already running:", session)
else:
    command = [sys.executable, "-u", str(EXPERIMENT/"scripts/prepare_data.py"), "--config", str(CONFIG)]
    shell_command = shlex.join(command) + " >> " + shlex.quote(str(log)) + " 2>&1"
    subprocess.run(["tmux", "new-session", "-d", "-s", session, shell_command], check=True, env=environment(cfg))
    print("Started. Log:", log)


## 준비 상태 확인
이 셀을 다시 실행해 로그를 갱신합니다. `PREPARATION COMPLETE` 또는 완료 marker를 확인한 후 2번 노트북으로 이동합니다.

In [ ]:
p = paths(cfg)
log = p["root"] / "prepare.console.log"
if log.exists():
    from collections import deque
    with log.open(errors="replace") as f:
        print("".join(deque(f, maxlen=25)))
marker = p["pre"] / cfg["dataset_name"] / "preparation.complete.json"
print("Preparation complete:", marker.exists())
if marker.exists():
    plans = json.loads((marker.parent / "nnUNetPlans.json").read_text())
    conf = plans["configurations"][cfg["configuration"]]
    print({k: conf.get(k) for k in ["patch_size", "batch_size", "spacing"]})
    splits = json.loads((marker.parent / "splits_final.json").read_text())
    print([(i, len(s["train"]), len(s["val"])) for i, s in enumerate(splits)])
